# Comparative Study: PPO vs DDPG vs SAC on CarRacing-v2

**Course:** Reinforcement Learning — Final Project  
**Environment:** `CarRacing-v2` (Gymnasium)  
**Algorithms:** PPO, DDPG, SAC (Stable-Baselines3)

---

## Research Questions

1. **Learning Speed:** Which method learns the fastest?
2. **Stability:** Which method is most stable across random seeds?
3. **Generalization:** Which method generalizes best to unseen tracks?
4. **Exploration:** What is the effect of exploration parameters?

---

## Experimental setup

We benchmark three deep reinforcement learning algorithms for continuous control:

- Proximal Policy Optimization (PPO)
- Deep Deterministic Policy Gradient (DDPG)
- Soft Actor-Critic (SAC)

The agents are trained on Gymnasium CarRacing using:

- 100,000 training timesteps
- 3 random seeds: 0, 1, and 2
- randomly generated tracks during training
- RGB image observations
- continuous steering, acceleration, and braking actions

Performance is evaluated through learning curves, seed-to-seed stability,
and generalization across randomly generated evaluation tracks.

In [ ]:
# Setup: add project root to path and import modules
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
from src.analysis import (
    load_training_curves,
    aggregate_curves,
    compute_time_to_threshold,
    load_all_eval_results,
    load_all_generalization_results,
    build_summary_table,
)
from src.plots import (
    plot_learning_curves,
    plot_time_to_threshold,
    plot_seed_stability_boxplot,
    plot_seed_curves,
    plot_generalization,
    plot_generalization_per_track,
    plot_exploration_effect,
    plot_exploration_comparison_bar,
)

%matplotlib inline
print("All modules loaded successfully.")

---

## 1. Learning Speed (Q1)

**Question:** Which algorithm learns the fastest on CarRacing-v2?

We compare learning curves (mean evaluation reward vs timesteps) and measure the number of timesteps required to reach a configurable reward threshold.

In [ ]:
# Load training curves for all algorithms
algos = ["ppo", "ddpg", "sac"]
all_curves = []
for algo in algos:
    try:
        curves = load_training_curves(algo)
        all_curves.append(curves)
    except FileNotFoundError:
        print(f"No training logs found for {algo} — run training first.")

if all_curves:
    curves_df = pd.concat(all_curves, ignore_index=True)
    agg_df = aggregate_curves(curves_df)
    print(f"Loaded curves for: {curves_df['algo'].unique().tolist()}")
    print(f"Total data points: {len(curves_df)}")
else:
    print("No training data available. Run scripts/run_all_training.py first.")

In [ ]:
# Plot comparative learning curves
if all_curves:
    fig = plot_learning_curves(curves_df, agg_df, save=False)
    plt.show()

In [ ]:
# Time to reach reward threshold
if all_curves:
    ttt_df = compute_time_to_threshold(curves_df)
    print("Timesteps to reach threshold:")
    print(ttt_df.to_string(index=False))
    print()
    fig = plot_time_to_threshold(ttt_df, save=False)
    plt.show()

---

## 2. Stability Across Seeds (Q2)

**Question:** Which algorithm is most stable (robust) across different random seeds?

We compare the variance in final evaluation performance across seeds using boxplots and per-seed learning curves.

In [ ]:
# Load evaluation results and display summary
try:
    eval_df = load_all_eval_results()
    summary = build_summary_table(eval_df)
    print("=== Performance Summary ===")
    print(summary[["algo", "variant", "reward_str", "n_seeds"]].to_string(index=False))
except FileNotFoundError:
    print("No evaluation results found. Run scripts/run_all_evaluation.py first.")
    eval_df = None

In [ ]:
# Stability boxplot
if eval_df is not None:
    default_eval = eval_df[eval_df["variant"] == "default"]
    if len(default_eval) > 0:
        fig = plot_seed_stability_boxplot(default_eval, save=False)
        plt.show()

In [ ]:
# Per-seed learning curves
if all_curves:
    fig = plot_seed_curves(curves_df, save=False)
    plt.show()

---

## 3. Generalization to Unseen Tracks (Q3)

**Question:** Which algorithm generalizes best to tracks never seen during training?

We evaluate each trained model on two sets of track seeds:
- **Seen tracks:** track seeds used during training (seeds 100–109)
- **Unseen tracks:** track seeds never encountered during training (seeds 200–209)

A model that generalizes well should show similar performance on both sets.

In [ ]:
# Load generalization results
try:
    gen_df = load_all_generalization_results()
    gen_summary = gen_df.groupby(["algo", "split"])["reward"].agg(["mean", "std", "count"]).reset_index()
    print("=== Generalization Summary ===")
    print(gen_summary.to_string(index=False))
except FileNotFoundError:
    print("No generalization results. Run: python scripts/run_all_evaluation.py --generalization")
    gen_df = None

In [ ]:
# Generalization plots
if gen_df is not None:
    fig = plot_generalization(gen_df, save=False)
    plt.show()
    
    fig = plot_generalization_per_track(gen_df, save=False)
    plt.show()

### Best Race Visualization on Test Tracks

This cell replays deterministic episodes for each algorithm on unseen test tracks and shows the best race found (highest episode reward).

Selection logic:
- If generalization results exist, select the best training seed for each algorithm on the **test** split.
- Otherwise, use seed 0 by default.

In [ ]:
# Visualize the best race for PPO/DDPG/SAC on unseen test tracks
from pathlib import Path
from matplotlib import animation
from IPython.display import HTML, display
from src.config import load_config
from src.env_factory import make_vec_env
from src.evaluate import load_model

algos = ["ppo", "ddpg", "sac"]

def select_best_seed_for_algo(algo, default_seed=0):
    if gen_df is None:
        return default_seed
    df = gen_df[(gen_df["algo"] == algo) & (gen_df["split"] == "test")]
    if "variant" in df.columns:
        df = df[df["variant"] == "default"]
    if df.empty:
        return default_seed
    return int(df.groupby("seed")["reward"].mean().idxmax())

def run_episode_and_collect_frames(model, env, max_steps=1000):
    obs = env.reset()
    done = False
    ep_reward = 0.0
    frames = []
    step = 0

    while not done and step < max_steps:
        frame = env.render()
        if frame is not None:
            frames.append(frame)

        action, _ = model.predict(obs, deterministic=True)
        obs, reward, dones, infos = env.step(action)
        ep_reward += float(reward[0])
        done = bool(dones[0])
        step += 1

    return ep_reward, frames

def show_frames_as_animation(frames, title, max_frames=250, stride=2):
    if not frames:
        print(f"No frames to display for {title}.")
        return

    sampled = frames[::stride][:max_frames]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis("off")
    ax.set_title(title)
    im = ax.imshow(sampled[0])

    def update(i):
        im.set_data(sampled[i])
        return (im,)

    ani = animation.FuncAnimation(fig, update, frames=len(sampled), interval=33, blit=True)
    display(HTML(ani.to_jshtml()))
    plt.close(fig)

summary_rows = []

for algo in algos:
    config = load_config(algo)
    model_root = Path(config["model_dir"])

    seed = select_best_seed_for_algo(algo, default_seed=0)
    model_path = model_root / f"{algo}_seed{seed}" / "best" / "best_model"

    if not model_path.with_suffix(".zip").exists():
        fallback_path = model_root / f"{algo}_seed{seed}" / "final_model"
        if fallback_path.with_suffix(".zip").exists():
            model_path = fallback_path
        else:
            print(f"[{algo.upper()}] No trained model found for seed={seed}.")
            continue

    model = load_model(algo, model_path)
    test_seeds = config.get("test_track_seeds", [200, 201, 202])[:3]

    best_reward = -1e9
    best_track_seed = None
    best_frames = None

    for track_seed in test_seeds:
        env = make_vec_env(
            env_id=config["env_id"],
            seed=int(track_seed),
            frame_stack=config["frame_stack"],
            render_mode="rgb_array",
        )

        ep_reward, frames = run_episode_and_collect_frames(model, env)
        env.close()

        if ep_reward > best_reward:
            best_reward = ep_reward
            best_track_seed = int(track_seed)
            best_frames = frames

    summary_rows.append({
        "algo": algo,
        "seed": seed,
        "track_seed": best_track_seed,
        "best_reward": best_reward,
        "model": str(model_path.with_suffix(".zip")),
    })

    title = f"{algo.upper()} | seed={seed} | best test track={best_track_seed} | reward={best_reward:.1f}"
    show_frames_as_animation(best_frames, title)

if summary_rows:
    summary_df = pd.DataFrame(summary_rows).sort_values("best_reward", ascending=False)
    print("=== Best Race Summary on Test Tracks ===")
    display(summary_df)

---

## 4. Effect of Exploration (Q4)

**Question:** How do exploration parameters affect learning?

We study the effect of:
- **PPO:** entropy coefficient (`ent_coef` = 0.0, 0.01, 0.05) — controls policy randomness
- **DDPG:** action noise intensity (`sigma` = 0.05, 0.2, 0.5) — Gaussian noise added to actions
- **SAC:** entropy coefficient (`ent_coef` = 0.05, auto, 0.2) — temperature parameter controlling exploration-exploitation balance

In [ ]:
# Exploration analysis: load variant curves per algorithm
for algo in algos:
    algo_config = load_config(algo)
    variants = algo_config.get("exploration_variants", {})
    if not variants:
        continue

    exploration_curves = {}
    for variant_name in variants:
        try:
            curves = load_training_curves(algo, exploration_variant=variant_name)
            agg = aggregate_curves(curves)
            exploration_curves[variant_name] = agg
        except FileNotFoundError:
            print(f"  No data for {algo}/{variant_name}")

    if exploration_curves:
        fig = plot_exploration_effect(exploration_curves, algo, save=False)
        plt.show()
    else:
        print(f"No exploration data for {algo}. Run scripts/run_exploration_experiments.py first.")

In [ ]:
# Exploration comparison bar chart (final performance)
try:
    results_dir = Path(load_config("ppo")["results_dir"]) / "evaluation"
    dfs = [pd.read_csv(p) for p in sorted(results_dir.glob("*_eval.csv"))]
    if dfs:
        all_eval = pd.concat(dfs, ignore_index=True)
        variant_eval = all_eval[all_eval["variant"] != "default"]
        if len(variant_eval) > 0:
            fig = plot_exploration_comparison_bar(variant_eval, save=False)
            plt.show()
except Exception as e:
    print(f"Could not generate exploration comparison: {e}")

## Conclusion

Under the training budget considered in this project, PPO achieved the
strongest overall performance and robustness across random tracks.

- PPO achieved a mean generalization reward of approximately 760.
- DDPG achieved approximately 687, but showed greater sensitivity to the
  training seed.
- SAC achieved approximately 611 and appeared to require a larger training
  budget to fully benefit from its exploration strategy.

Overall, PPO provided the best short-budget trade-off between learning
performance, stability, and robustness in CarRacing.

### Summary table

In [ ]:
# Final summary table
if eval_df is not None:
    summary = build_summary_table(eval_df[eval_df["variant"] == "default"])
    print("=" * 60)
    print("FINAL PERFORMANCE COMPARISON (default hyperparameters)")
    print("=" * 60)
    print(summary[["algo", "reward_str", "n_seeds"]].to_string(index=False))
    print()

# Generalization gap
if gen_df is not None:
    gen_gap = gen_df.groupby(["algo", "split"])["reward"].mean().unstack("split")
    gen_gap["gap"] = gen_gap["train"] - gen_gap["test"]
    gen_gap["gap_%"] = (gen_gap["gap"] / gen_gap["train"] * 100).round(1)
    print("=" * 60)
    print("GENERALIZATION GAP (seen - unseen tracks)")
    print("=" * 60)
    print(gen_gap.to_string())
    print()

# Learning speed
if all_curves:
    ttt_summary = ttt_df.groupby("algo")["timesteps_to_threshold"].agg(["mean", "std"])
    print("=" * 60)
    print(f"LEARNING SPEED (timesteps to reach threshold {ttt_df['threshold'].iloc[0]})")
    print("=" * 60)
    print(ttt_summary.to_string())